# Topic Discovery Menggunakan LDA (Ulasan Aplikasi)
Eksplorasi topik laten dari ulasan pelanggan Kopi Kenangan di Play Store menggunakan Latent Dirichlet Allocation (LDA) untuk memvalidasi taksonomi aspek.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import joblib

## 1. Load Data

In [ ]:
dataset_path = '../../preprocessing/cleaned_app_reviews.csv'
df = pd.read_csv(dataset_path).dropna(subset=['text_clean'])
print(f"Total data: {len(df)} baris")
df.head()

## 2. Vektorisasi TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000, 
    ngram_range=(1, 2), 
    min_df=3, 
    max_df=0.9
)
tfidf_matrix = vectorizer.fit_transform(df['text_clean'])
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

## 3. Training LDA Model
Inisialisasi LDA dengan n_components=8 sesuai dengan jumlah aspek taksonomi yang didefinisikan untuk aplikasi.

In [ ]:
n_topics = 8
lda = LatentDirichletAllocation(
    n_components=n_topics, 
    max_iter=15, 
    learning_method='online', 
    random_state=42,
    n_jobs=-1
)
lda.fit(tfidf_matrix)

## 4. Tampilkan Topik Laten
Menampilkan 15 kata teratas dari setiap topik.

In [ ]:
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[:-15 - 1:-1]
    top_words = [feature_names[i] for i in top_words_idx]
    print(f"Topic #{topic_idx + 1}:")
    print(" | ".join(top_words))
    print("-" * 80)

### Pemetaan Topik ke Aspek:
Silakan periksa hasil kata kunci per topik di atas dan petakan ke aspek taksonomi berikut:
- `aspek_rasa` (Rasa & Kualitas Minuman)
- `aspek_harga` (Harga & Promo)
- `aspek_aplikasi` (Performa & Fitur Aplikasi)
- `aspek_pelayanan` (Pelayanan CS/Support)
- `aspek_kecepatan` (Kecepatan & Waktu Tunggu)
- `aspek_stok` (Ketersediaan Stok/Menu)
- `aspek_pengiriman` (Layanan Pengiriman)
- `aspek_pesanan` (Akurasi & Sistem Pesanan)

## 5. Distribusi Topik per Ulasan

In [ ]:
topic_distributions = lda.transform(tfidf_matrix)
df['predicted_topic'] = topic_distributions.argmax(axis=1)
df['topic_probability'] = topic_distributions.max(axis=1)

# Tampilkan contoh review dengan probabilitas tertinggi per topik
for t in range(n_topics):
    print(f"\n--- Contoh Ulasan untuk Topic #{t + 1} ---")
    samples = df[df['predicted_topic'] == t].sort_values(by='topic_probability', ascending=False).head(3)
    for idx, row in samples.iterrows():
        print(f"- {row['review_text']} (Prob: {row['topic_probability']:.2f})")

## 6. Simpan Model

In [ ]:
os.makedirs('../weights', exist_ok=True)
os.makedirs('../data', exist_ok=True)

df.to_csv('../data/reviews_with_topics.csv', index=False)
joblib.dump(lda, '../weights/lda_topic_model.pkl')
joblib.dump(vectorizer, '../weights/tfidf_vectorizer_lda.pkl')
print("Model LDA dan vectorizer berhasil disimpan!")